# Kafka 

Apache Kafka is an open-source, distributed event streaming platform used to collect, process, store, and route large volumes of real-time data.

It was developed by LinkedIn and later donated to the Apache Software Foundation.

## The Problem

Let's say we have an e-commerce platform where multiple microservices communicate with each other. 

![image_alt_text](../img/microservices.png)

This works well, until the user base becomes huge. There are a few problems we incur at scale:


!["tight coupling"](../img/microservices_issues.png)


In short:

Each service waits for certain responses from different services to proceed further with their actions. 

1. If one service fails, multiple services might fail as a consequence
2. If one service is slow, it will slow down the entire system and users will see indefinite delays.
3. We also lose data we could have otherwise stored and used for analytics.


## The Solution



Instead of having microservices communicate directly with one another, we structure their requests and responses as events and use Kafka as the central event-streaming layer. 

Services publish events to Kafka and consume the events they need, enabling asynchronous, decoupled communication between services.


Consider an e-commerce platform with these microservices:

```text
User Service
Product Service
Order Service
Payment Service
Inventory Service
Notification Service
Shipping Service
```

### Without Kafka — direct communication

Suppose a customer places an order.

The `Order Service` might directly call several other services:

```text
Customer
   │
   ↓
Order Service
   │
   ├──→ Payment Service
   │
   ├──→ Inventory Service
   │
   ├──→ Notification Service
   │
   └──→ Shipping Service
```

The Order Service now needs to know **where and how to communicate with all these services**.

If the Notification Service is temporarily down, it can also complicate the order flow.

---

### With Kafka — event-driven communication

Instead, the Order Service publishes an event:

```json
{
    "event": "OrderPlaced",
    "order_id": 12345,
    "user_id": 101,
    "amount": 2499,
    "items": [
        {
            "product_id": 501,
            "quantity": 2
        }
    ]
}
```

It publishes this event to Kafka:

```text
                    ┌─────────────────┐
                    │   Order Service │
                    └────────┬────────┘
                             │
                             │ OrderPlaced
                             ↓
                       ┌───────────┐
                       │   Kafka   │
                       └─────┬─────┘
                             │
              ┌──────────────┼──────────────┐
              ↓              ↓              ↓
       Payment Service  Inventory       Notification
                         Service           Service
```

Each service consumes the event independently.

For example:

**Payment Service**

```text
OrderPlaced
     ↓
Process payment
     ↓
PaymentCompleted
```

**Inventory Service**

```text
OrderPlaced
     ↓
Reserve inventory
     ↓
InventoryReserved
```

**Notification Service**

```text
OrderPlaced
     ↓
Send "Order received" email
```

And those resulting events can go back into Kafka:

```text
                    Kafka
                      ↑
                      │
        ┌─────────────┼─────────────┐
        │             │             │
        ↓             ↓             ↓
PaymentCompleted  InventoryReserved  NotificationSent
```

Other services can consume these events as needed.

### The key difference

Instead of:

```text
Order Service → Payment Service
Order Service → Inventory Service
Order Service → Notification Service
Order Service → Shipping Service
```

you have:

```text
                         Kafka
                           │
                    OrderPlaced
                           │
             ┌─────────────┼─────────────┐
             ↓             ↓             ↓
          Payment      Inventory      Notification
          Service       Service          Service
             │             │
             ↓             ↓
      PaymentCompleted  InventoryReserved
```

So the services **don't need to know about each other's APIs directly**. They communicate through events published to Kafka.

The important distinction is that Kafka isn't really "managing requests and responses" like an API gateway. **Kafka is carrying events**, while each microservice decides how to react to those events.

This is what people mean by **event-driven architecture** and **loosely coupled microservices**.


## Core Kafka Concepts

**Topic** — a named stream of events, like a category. In Nabz, you'll have a topic like `raw-feedback`. Think of it as a named log file that things get appended to.

**Partition** — a topic is split into partitions for scale. Each partition is an ordered, append-only sequence of messages. Order is guaranteed *within* a partition, but not *across* partitions of the same topic. This is the detail that trips people up — Kafka is not "one big ordered queue," it's several ordered queues (partitions) that together make up a topic.

**Offset** — each message in a partition gets a sequential ID (0, 1, 2, 3...). Consumers track which offset they've read up to, so they know where to resume.

**Producer** (service generating an event) — writes messages to a topic. It can specify a key; messages with the same key always land in the same partition (this matters — e.g., if you key by `user_id`, all of one user's feedback stays in order).

**Consumer** (service receiving an event) — reads messages from a topic, starting from wherever it last left off (its offset).

**Consumer group** — a set of consumers working together to read a topic. Kafka splits the partitions among them (e.g., 4 partitions, 2 consumers → each consumer gets 2 partitions). This is how Kafka scales reads — add more consumers to a group, get more parallelism, up to the number of partitions.

**Broker** — a single Kafka server. A **cluster** is multiple brokers working together, replicating data so if one dies, others have copies.


## How it actually works under the hood

!["Kafka Architecture"](../img/kafka_architecture.png)

**Kafka is fundamentally a log, not a queue.** A traditional message queue (RabbitMQ, SQS) deletes a message once it's consumed. Kafka doesn't — messages stay on disk for a configured retention period (say, 7 days) regardless of whether anyone's read them. This is a deliberate design choice: it means multiple independent consumers can read the same data at different speeds, and you can replay history by resetting a consumer's offset backward. This is huge for debugging — if your sentiment scoring logic has a bug, you can rewind and reprocess without re-ingesting anything.

**Writes are append-only and sequential.** This is why Kafka is so fast despite writing to disk — sequential disk writes are nearly as fast as memory, and Kafka leans hard into that instead of fighting it with random-access patterns.

**Replication for durability.** Each partition has a "leader" broker and some number of "follower" replicas on other brokers. Producers write to the leader; followers copy the data. If the leader's broker dies, a follower is promoted, and no data is lost (as long as it was replicated before the crash).